# NiyamTrace-X — Final Multi-Provider Paper Closure

This notebook removes OpenRouter from the critical path.

It securely asks for:
- **OpenAI API key**
- **Gemini API key**
- **Groq API key**

Then it selects **three independent tool-capable model families** from the working providers.

Preferred closure matrix:
- OpenAI → OpenAI GPT family
- Gemini → Gemini family
- Groq → Qwen family

Fallback when OpenAI API quota/billing is unavailable:
- Gemini → Gemini
- Groq → Qwen
- Groq → Llama or GPT-OSS

## Required external benchmarks

1. BFCL-v4
2. AgentDojo
3. τ³ / tau2-bench

## Closure mode

Default `CLOSURE` is intentionally bounded enough to fit free/low-cost API quotas while still yielding real external evidence:

- BFCL: 25 cases/model
- AgentDojo: one controlled task/injection pair over four suites
- τ³: 3 tasks/domain × 3 domains × 3 families

Every output, error, trajectory, score, environment snapshot, and manifest is archived.

No benchmark is marked supported unless native numeric outputs exist.

In [ ]:
# CELL 1 — CORE SETUP
from pathlib import Path
from getpass import getpass
from datetime import datetime, timezone
import os,sys,json,re,time,random,hashlib,zipfile,shutil,subprocess,platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED=42
random.seed(SEED); np.random.seed(SEED)

SELFTEST=os.getenv("NTX_SELFTEST","0")=="1"
MODE=os.getenv("NTX_MULTI_PROVIDER_MODE","CLOSURE").upper()
assert MODE in {"SMOKE","CLOSURE","FULL"}

BASE=Path(os.getenv(
    "NTX_MULTI_PROVIDER_DIR",
    "/content/NTX_MULTI_PROVIDER_CLOSURE" if Path("/content").exists() else str(Path.cwd()/"NTX_MULTI_PROVIDER_CLOSURE")
))
WORK=BASE/"work"; RAW=BASE/"raw"; RESULTS=BASE/"results"; LOGS=BASE/"logs"
ENV=BASE/"environment"; PAPER=BASE/"paper_integration"; ARCH=BASE/"archives"
for p in [BASE,WORK,RAW,RESULTS,LOGS,ENV,PAPER,ARCH]:p.mkdir(parents=True,exist_ok=True)

CFG={
    "SMOKE":{
        "bfcl_limit":3,
        "dojo_suites":["banking"],
        "dojo_user_tasks":["user_task_0"],
        "dojo_injection_tasks":["injection_task_0"],
        "tau_domains":["airline","retail","telecom"],
        "tau_tasks":1,
        "tau_steps":16,
        "tau_agent_tokens":768,
        "tau_user_tokens":384,
        "bootstrap":500,
    },
    "CLOSURE":{
        "bfcl_limit":25,
        "dojo_suites":["banking","workspace","travel","slack"],
        "dojo_user_tasks":["user_task_0"],
        "dojo_injection_tasks":["injection_task_0"],
        "tau_domains":["airline","retail","telecom"],
        "tau_tasks":3,
        "tau_steps":35,
        "tau_agent_tokens":1024,
        "tau_user_tokens":512,
        "bootstrap":3000,
    },
    "FULL":{
        "bfcl_limit":None,
        "dojo_suites":["banking","workspace","travel","slack"],
        "dojo_user_tasks":None,
        "dojo_injection_tasks":None,
        "tau_domains":["airline","retail","telecom"],
        "tau_tasks":None,
        "tau_steps":80,
        "tau_agent_tokens":2048,
        "tau_user_tokens":1024,
        "bootstrap":10000,
    }
}[MODE]
print("MODE:",MODE)
print(json.dumps(CFG,indent=2))

In [ ]:
# CELL 2 — HELPERS
CHECKPOINT=RESULTS/"checkpoint.json"

def now(): return datetime.now(timezone.utc).isoformat()

def run(cmd,cwd=None,env=None,timeout=None):
    return subprocess.run(
        [str(x) for x in cmd],cwd=str(cwd) if cwd else None,env=env,
        capture_output=True,text=True,errors="replace",timeout=timeout
    )

def save_log(name,p,cmd=None):
    parts=[]
    if cmd is not None:parts+=["COMMAND"," ".join(map(str,cmd)),""]
    parts+=["STDOUT",p.stdout or "","STDERR",p.stderr or ""]
    (LOGS/name).write_text("\n".join(parts),errors="ignore")

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for c in iter(lambda:f.read(1024*1024),b""):h.update(c)
    return h.hexdigest()

def save_json(path,obj):Path(path).write_text(json.dumps(obj,indent=2,default=str))

def load_json(path,default=None):
    try:return json.loads(Path(path).read_text())
    except Exception:return {} if default is None else default

STATE=load_json(CHECKPOINT,{})
def ck_get(k):return STATE.get(k,{})
def ck_set(k,status,**extra):
    STATE[k]={"status":status,"updated_at":now(),**extra};save_json(CHECKPOINT,STATE)
def supported(k):return ck_get(k).get("status")=="SUPPORTED"

def ensure_uv():
    if not shutil.which("uv"):
        p=run([sys.executable,"-m","pip","install","-q","-U","uv"])
        if p.returncode:raise RuntimeError("uv installation failed")

def clone(url,dest):
    dest=Path(dest)
    if not dest.exists():
        p=run(["git","clone","--depth","1",url,dest]);save_log("clone_"+dest.name+".log",p)
        if p.returncode:raise RuntimeError("clone failed: "+url)
    return run(["git","-C",dest,"rev-parse","HEAD"]).stdout.strip()

def copytree(src,dst):
    src,dst=Path(src),Path(dst)
    if not src.exists():return False
    shutil.rmtree(dst,ignore_errors=True);shutil.copytree(src,dst);return True

def make_zip(src,dst):
    src,dst=Path(src),Path(dst)
    if dst.exists():dst.unlink()
    with zipfile.ZipFile(dst,"w",zipfile.ZIP_DEFLATED,allowZip64=True) as z:
        for p in sorted(src.rglob("*")):
            if p.is_file():z.write(p,arcname=str(p.relative_to(src)))
    return dst

def error_class(text):
    t=(text or "").lower()
    if any(x in t for x in ["402","insufficient_quota","billing","credits","quota"]):return "BILLING_OR_QUOTA"
    if any(x in t for x in ["429","rate limit","resource_exhausted"]):return "RATE_LIMIT"
    if any(x in t for x in ["404","not found","model_not_found"]):return "MODEL_UNAVAILABLE"
    if any(x in t for x in ["modulenotfound","importerror"]):return "DEPENDENCY"
    return "OTHER"

In [ ]:
# CELL 3 — INSTALL/IMPORT OPENAI-COMPATIBLE CLIENT + ASK THREE KEYS
if SELFTEST:
    OpenAI=None
else:
    try:
        from openai import OpenAI
    except Exception:
        p=run([sys.executable,"-m","pip","install","-q","-U","openai"])
        save_log("install_openai_client.log",p)
        if p.returncode:
            print(p.stderr)
            raise RuntimeError("Could not install the openai Python client.")
        from openai import OpenAI

def ask_secret(name,label):
    v=os.getenv(name,"").strip()
    if not v and not SELFTEST:
        v=getpass(label+": ").strip()
    if v:
        os.environ[name]=v
    return v

if SELFTEST:
    OPENAI_API_KEY=GEMINI_API_KEY=GROQ_API_KEY="SELFTEST"
else:
    OPENAI_API_KEY=ask_secret("OPENAI_API_KEY","OpenAI API key (hidden; Enter to skip)")
    GEMINI_API_KEY=ask_secret("GEMINI_API_KEY","Gemini API key (hidden; Enter to skip)")
    GROQ_API_KEY=ask_secret("GROQ_API_KEY","Groq API key (hidden; Enter to skip)")

PROVIDERS={
    "openai":{"key":OPENAI_API_KEY,"base_url":"https://api.openai.com/v1"},
    "gemini":{"key":GEMINI_API_KEY,"base_url":"https://generativelanguage.googleapis.com/v1beta/openai/"},
    "groq":{"key":GROQ_API_KEY,"base_url":"https://api.groq.com/openai/v1"},
}
print({k:("SET" if v["key"] else "SKIPPED") for k,v in PROVIDERS.items()})


In [ ]:
# CELL 4 — CANDIDATE MODEL FAMILIES + TOOL PREFLIGHT
CANDIDATES=[
    {"provider":"openai","family":"OpenAI-GPT","model":"gpt-5.6-luna","priority":1},
    {"provider":"openai","family":"OpenAI-GPT","model":"gpt-5.6-terra","priority":2},
    {"provider":"gemini","family":"Gemini","model":"gemini-3.8-flash","priority":1},
    {"provider":"groq","family":"Qwen","model":"qwen/qwen3.6-27b","priority":1},
    {"provider":"groq","family":"Llama","model":"llama-3.1-8b-instant","priority":1},
    {"provider":"groq","family":"Llama","model":"llama-3.3-70b-versatile","priority":2},
    {"provider":"groq","family":"GPT-OSS","model":"openai/gpt-oss-20b","priority":1},
]

tool=[{"type":"function","function":{
    "name":"lookup_order","description":"Look up an order",
    "parameters":{"type":"object","properties":{"order_id":{"type":"string"}},"required":["order_id"]}
}}]

rows=[]
if SELFTEST:
    for c in [CANDIDATES[0],CANDIDATES[2],CANDIDATES[3]]:
        rows.append({**c,"chat_ok":True,"tool_ok":True,"status":"SELFTEST_ONLY"})
else:
    for c in CANDIDATES:
        cfg=PROVIDERS[c["provider"]]
        if not cfg["key"]:continue
        client=OpenAI(api_key=cfg["key"],base_url=cfg["base_url"])
        r=dict(c)
        try:
            a=client.chat.completions.create(
                model=c["model"],messages=[{"role":"user","content":"Reply exactly OK"}],max_tokens=16
            )
            r["chat_ok"]=bool(a.choices);r["chat_error"]=""
        except Exception as e:
            r["chat_ok"]=False;r["chat_error"]=repr(e)
        try:
            b=client.chat.completions.create(
                model=c["model"],
                messages=[{"role":"user","content":"Use lookup_order for order A123."}],
                tools=tool,tool_choice="auto",max_tokens=96
            )
            msg=b.choices[0].message if b.choices else None
            r["tool_ok"]=bool(getattr(msg,"tool_calls",None));r["tool_error"]=""
        except Exception as e:
            r["tool_ok"]=False;r["tool_error"]=repr(e)
        r["status"]="OK" if r["chat_ok"] and r["tool_ok"] else "FAILED"
        rows.append(r)

preflight=pd.DataFrame(rows)
preflight.to_csv(RESULTS/"00_provider_model_preflight.csv",index=False)
display(preflight)

In [ ]:
# CELL 5 — SELECT THREE INDEPENDENT FAMILIES
# Provider diversity first; then Groq fallbacks if OpenAI is unavailable.
priority=[
    ("openai","OpenAI-GPT"),
    ("gemini","Gemini"),
    ("groq","Qwen"),
    ("groq","Llama"),
    ("groq","GPT-OSS"),
]

selected=[]
used=set()
for provider,family in priority:
    hit=preflight[
        (preflight.provider==provider)&
        (preflight.family==family)&
        (preflight.status.isin(["OK","SELFTEST_ONLY"]))
    ].sort_values("priority")
    if len(hit) and family not in used:
        r=hit.iloc[0]
        selected.append({
            "provider":provider,"family":family,"model":r.model,
            "base_url":PROVIDERS[provider]["base_url"],
        })
        used.add(family)
    if len(selected)>=3:break

selected_df=pd.DataFrame(selected)
selected_df.to_csv(RESULTS/"01_selected_models.csv",index=False)
display(selected_df)

if len(selected)<3 and not SELFTEST:
    raise RuntimeError(
        "Fewer than three independent tool-capable families are available. "
        "Check the failing provider key/quota, then rerun this cell."
    )

# BFCL-v4

Each selected provider/model is run against the same BFCL-v4 interface through EvalScope. Provider base URL and API key are passed independently per run.

In [ ]:
# CELL 6 — BFCL ENVIRONMENT
BFENV=WORK/"bfcl_env"
if SELFTEST:
    BFPY=Path(sys.executable)
else:
    ensure_uv();run(["uv","python","install","3.11"])
    if not BFENV.exists():
        p=run(["uv","venv",BFENV,"--python","3.11"])
        if p.returncode:raise RuntimeError("BFCL venv creation failed")
    BFPY=BFENV/"bin"/"python"
    p=run(["uv","pip","install","--python",BFPY,"-U","evalscope[bfcl]"])
    save_log("bfcl_install.log",p)
    if p.returncode:raise RuntimeError("BFCL install failed")
    v=run([BFPY,"-c","from evalscope import run_task; from evalscope.config import TaskConfig; print('OK')"])
    if v.returncode:raise RuntimeError("BFCL import verification failed")
print("BFCL environment ready")

In [ ]:
# CELL 7 — BFCL RUN + PARSE
def parse_bfcl(root):
    root=Path(root);vals=[];evaluated=0
    for p in root.rglob("*"):
        if not p.is_file():continue
        rel=str(p.relative_to(root));low=rel.lower()
        try:
            if p.suffix.lower()==".csv":
                df=pd.read_csv(p)
                if any(x in low for x in ["report","review","prediction"]):evaluated=max(evaluated,len(df))
                for c in df.columns:
                    if any(x in str(c).lower() for x in ["accuracy","score"]):
                        for v in pd.to_numeric(df[c],errors="coerce").dropna():
                            vals.append({"slice":rel,"metric":str(c),"score":float(v)})
            elif p.suffix.lower() in {".json",".jsonl"}:
                texts=p.read_text(errors="ignore").splitlines() if p.suffix.lower()==".jsonl" else [p.read_text(errors="ignore")]
                for txt in texts:
                    try:o=json.loads(txt)
                    except Exception:continue
                    stack=[o]
                    while stack:
                        x=stack.pop()
                        if isinstance(x,dict):
                            for ck in ["total_count","evaluated_count","num_samples","n_samples"]:
                                if isinstance(x.get(ck),(int,float)) and x[ck]>0:evaluated=max(evaluated,int(x[ck]))
                            for k,v in x.items():
                                if isinstance(v,(dict,list)):stack.append(v)
                                elif isinstance(v,(int,float)) and any(q in k.lower() for q in ["accuracy","score"]):
                                    vals.append({"slice":rel,"metric":k,"score":float(v)})
                        elif isinstance(x,list):stack.extend(x)
        except Exception:pass
    return pd.DataFrame(vals).drop_duplicates() if vals else pd.DataFrame(),evaluated

bfcl_rows=[];bfcl_status=[]
if SELFTEST:
    for s in selected:
        root=RAW/"bfcl"/s["family"];root.mkdir(parents=True,exist_ok=True)
        (root/"report.json").write_text(json.dumps({"accuracy":0.8,"total_count":5}))
        d,n=parse_bfcl(root);d["family"]=s["family"];d["model"]=s["model"];d["provider"]=s["provider"]
        bfcl_rows.append(d);bfcl_status.append({"family":s["family"],"status":"SELFTEST_ONLY","evaluated":n})
else:
    for s in selected:
        key=f"bfcl::{s['family']}::{MODE}"
        root=RAW/"bfcl"/s["family"]/MODE.lower();root.mkdir(parents=True,exist_ok=True)
        if supported(key):
            d,n=parse_bfcl(root)
            if n>0 and len(d):
                d["family"]=s["family"];d["model"]=s["model"];d["provider"]=s["provider"]
                bfcl_rows.append(d);bfcl_status.append({"family":s["family"],"status":"SUPPORTED_REUSED","evaluated":n})
                continue

        api_key=PROVIDERS[s["provider"]]["key"]
        script=root/"run.py"
        script.write_text(
f'''from evalscope import run_task
from evalscope.config import TaskConfig
cfg=TaskConfig(
 model={s["model"]!r},
 api_url={s["base_url"]!r},
 api_key={api_key!r},
 eval_type="openai_api",
 datasets=["bfcl_v4"],
 work_dir={str(root)!r},
 limit={CFG["bfcl_limit"]!r},
 seed={SEED},
 generation_config={{"temperature":0.1,"max_tokens":1024,"retries":1,"timeout":120}},
 dataset_args={{"bfcl_v4":{{"extra_params":{{"is_fc_model":True}}}}}}
)
run_task(task_cfg=cfg)
''')
        p=run([BFPY,script],cwd=root,timeout=None);save_log("bfcl_"+s["family"]+".log",p,[BFPY,script])
        d,n=parse_bfcl(root)
        if len(d):
            d["family"]=s["family"];d["model"]=s["model"];d["provider"]=s["provider"];bfcl_rows.append(d)
        st="SUPPORTED" if n>0 and len(d) else "INFRA_FAILURE"
        ck_set(key,st,evaluated=n,returncode=p.returncode,error_class=error_class((p.stdout or "")+(p.stderr or "")))
        bfcl_status.append({"family":s["family"],"status":st,"evaluated":n,"returncode":p.returncode})

bfcl=pd.concat(bfcl_rows,ignore_index=True) if bfcl_rows else pd.DataFrame()
bfcl_stat=pd.DataFrame(bfcl_status)
bfcl.to_csv(RESULTS/"10_bfcl_metrics.csv",index=False);bfcl_stat.to_csv(RESULTS/"10_bfcl_status.csv",index=False)
display(bfcl_stat)

# AgentDojo

Every model is passed through AgentDojo's `openai-compatible` adapter using that provider's own base URL and key.

In [ ]:
# CELL 8 — AGENTDOJO ENVIRONMENT
DOJO=WORK/"agentdojo"
if SELFTEST:
    DOJO_COMMIT="SELFTEST"
else:
    DOJO_COMMIT=clone("https://github.com/ethz-spylab/agentdojo.git",DOJO)
    ensure_uv()
    p=run(["uv","sync"],cwd=DOJO);save_log("dojo_sync.log",p)
    if p.returncode:raise RuntimeError("AgentDojo install failed")
    h=run(["uv","run","python","-m","agentdojo.scripts.benchmark","--help"],cwd=DOJO)
    txt=(h.stdout or "")+(h.stderr or "")
    if not all(x in txt for x in ["openai-compatible","--model-id","--force-rerun"]):
        raise RuntimeError("AgentDojo current CLI adapter not available")
print("AgentDojo ready")

In [ ]:
# CELL 9 — AGENTDOJO RUN + PARSE
def parse_dojo(root,s):
    rows=[]
    for p in Path(root).rglob("*.json"):
        try:o=json.loads(p.read_text())
        except Exception:continue
        if not isinstance(o,dict):continue
        u=o.get("utility");sec=o.get("security")
        if not isinstance(u,bool) and not isinstance(sec,bool):continue
        rows.append({
            "provider":s["provider"],"family":s["family"],"model":s["model"],
            "utility":np.nan if not isinstance(u,bool) else int(u),
            "security":np.nan if not isinstance(sec,bool) else int(sec),
            "error":o.get("error"),"user_task_id":o.get("user_task_id"),
            "injection_task_id":o.get("injection_task_id"),"source_file":str(p)
        })
    return pd.DataFrame(rows)

dojo_parts=[];dojo_status=[]
if SELFTEST:
    for s in selected:
        for suite in CFG["dojo_suites"]:
            d=pd.DataFrame([{"provider":s["provider"],"family":s["family"],"model":s["model"],
                             "utility":1,"security":1,"error":None,"user_task_id":"user_task_0",
                             "injection_task_id":"injection_task_0","source_file":"SELFTEST","suite":suite}])
            dojo_parts.append(d);dojo_status.append({"family":s["family"],"suite":suite,"status":"SELFTEST_ONLY","n":1})
else:
    for s in selected:
        for suite in CFG["dojo_suites"]:
            key=f"dojo::{s['family']}::{suite}::{MODE}"
            root=RAW/"agentdojo"/s["family"]/suite/MODE.lower();root.mkdir(parents=True,exist_ok=True)
            if supported(key):
                d=parse_dojo(root,s)
                if len(d):
                    d["suite"]=suite;dojo_parts.append(d);dojo_status.append({"family":s["family"],"suite":suite,"status":"SUPPORTED_REUSED","n":len(d)})
                    continue

            env=os.environ.copy()
            env["OPENAI_COMPATIBLE_BASE_URL"]=s["base_url"]
            env["OPENAI_COMPATIBLE_API_KEY"]=PROVIDERS[s["provider"]]["key"]
            cmd=["uv","run","python","-m","agentdojo.scripts.benchmark",
                 "--model","openai-compatible","--model-id",s["model"],
                 "-s",suite,"--attack","important_instructions",
                 "--logdir",str(root),"--force-rerun","--max-workers","1"]
            if CFG["dojo_user_tasks"] is not None:
                for ut in CFG["dojo_user_tasks"]:cmd+=["-ut",ut]
            if CFG["dojo_injection_tasks"] is not None:
                for it in CFG["dojo_injection_tasks"]:cmd+=["-it",it]

            p=run(cmd,cwd=DOJO,env=env,timeout=None);save_log(f"dojo_{s['family']}_{suite}.log",p,cmd)
            d=parse_dojo(root,s)
            if len(d):d["suite"]=suite;dojo_parts.append(d)
            valid=int(((d.utility.notna())|(d.security.notna())).sum()) if len(d) else 0
            errors=int(d.error.notna().sum()) if len(d) else 0
            st="SUPPORTED" if valid>0 and errors==0 else ("PARTIAL" if valid>0 else "INFRA_FAILURE")
            ck_set(key,st,n=valid,errors=errors,returncode=p.returncode)
            dojo_status.append({"family":s["family"],"suite":suite,"status":st,"n":valid,"errors":errors})

dojo=pd.concat(dojo_parts,ignore_index=True) if dojo_parts else pd.DataFrame()
dojo_stat=pd.DataFrame(dojo_status)
dojo.to_csv(RESULTS/"11_agentdojo_cases.csv",index=False);dojo_stat.to_csv(RESULTS/"11_agentdojo_status.csv",index=False)
display(dojo_stat)

# τ³ / tau2-bench

τ³ uses LiteLLM, so each provider uses its native LiteLLM route:

- OpenAI → `openai/<model>`
- Gemini → `gemini/<model>`
- Groq → `groq/<model>`

All three API keys are passed only through environment variables.

In [ ]:
# CELL 10 — TAU ENVIRONMENT
TAU=WORK/"tau2-bench"
if SELFTEST:
    TAU_COMMIT="SELFTEST"
else:
    TAU_COMMIT=clone("https://github.com/sierra-research/tau2-bench.git",TAU)
    ensure_uv()
    p=run(["uv","sync"],cwd=TAU);save_log("tau_sync.log",p)
    if p.returncode:raise RuntimeError("tau2 install failed")
    TAUPY=TAU/".venv"/"bin"/"python"
    d=run(["uv","pip","install","--python",TAUPY,"websockets","soundfile"],cwd=TAU);save_log("tau_deps.log",d)
    v=run([TAUPY,"-c","import websockets,soundfile;print('OK')"],cwd=TAU)
    if v.returncode:raise RuntimeError("tau2 dependency verification failed")
print("tau2 ready")

In [ ]:
# CELL 11 — TAU PROVIDER MAPPING
def tau_model(s):
    if s["provider"]=="openai":return "openai/"+s["model"]
    if s["provider"]=="gemini":return "gemini/"+s["model"]
    if s["provider"]=="groq":return "groq/"+s["model"]
    raise ValueError(s)

# Use cheapest available selected model as user simulator.
# Prefer Groq Llama/Qwen, then Gemini, then OpenAI.
pref_order={"groq":0,"gemini":1,"openai":2}
fixed_user=sorted(selected,key=lambda x:pref_order.get(x["provider"],9))[0]
FIXED_USER_TAU=tau_model(fixed_user)
print("Fixed tau user simulator:",fixed_user["family"],FIXED_USER_TAU)

In [ ]:
# CELL 12 — TAU RUN + STRICT PARSE
def parse_tau(path):
    try:o=json.loads(Path(path).read_text())
    except Exception:return pd.DataFrame()
    sims=o if isinstance(o,list) else next((o[k] for k in ["simulations","results","trajectories"] if isinstance(o,dict) and isinstance(o.get(k),list)),[])
    rows=[]
    for i,s in enumerate(sims):
        if not isinstance(s,dict):continue
        reward=None
        if isinstance(s.get("reward_info"),dict) and isinstance(s["reward_info"].get("reward"),(int,float,bool)):
            reward=float(s["reward_info"]["reward"])
        elif isinstance(s.get("reward"),(int,float,bool)):reward=float(s["reward"])
        err=s.get("error")
        if err is None and isinstance(s.get("info"),dict):err=s["info"].get("error")
        rows.append({"trajectory_index":i,"task_id":s.get("task_id"),"reward":reward,"error":err})
    return pd.DataFrame(rows)

tau_parts=[];tau_status=[]
if SELFTEST:
    for s in selected:
        for domain in CFG["tau_domains"]:
            d=pd.DataFrame([{"trajectory_index":0,"task_id":"0","reward":1.0,"error":None},
                            {"trajectory_index":1,"task_id":"1","reward":0.0,"error":None}])
            d["provider"]=s["provider"];d["family"]=s["family"];d["model"]=s["model"];d["domain"]=domain
            tau_parts.append(d);tau_status.append({"family":s["family"],"domain":domain,"status":"SELFTEST_ONLY","n":2})
else:
    env=os.environ.copy()
    if OPENAI_API_KEY:env["OPENAI_API_KEY"]=OPENAI_API_KEY
    if GEMINI_API_KEY:env["GEMINI_API_KEY"]=GEMINI_API_KEY
    if GROQ_API_KEY:env["GROQ_API_KEY"]=GROQ_API_KEY

    for s in selected:
        for domain in CFG["tau_domains"]:
            key=f"tau::{s['family']}::{domain}::{MODE}"
            run_name=f"ntx_multi_{s['family'].lower().replace('-','_')}_{domain}_{MODE.lower()}"
            live=TAU/"data"/"simulations"/run_name
            archive=RAW/"tau"/s["family"]/domain/MODE.lower()

            if supported(key) and (archive/"results.json").exists():
                d=parse_tau(archive/"results.json")
                if d.reward.notna().any():
                    d["provider"]=s["provider"];d["family"]=s["family"];d["model"]=s["model"];d["domain"]=domain
                    tau_parts.append(d);tau_status.append({"family":s["family"],"domain":domain,"status":"SUPPORTED_REUSED","n":int(d.reward.notna().sum())})
                    continue

            cmd=["uv","run","tau2","run","--domain",domain,
                 "--agent-llm",tau_model(s),"--user-llm",FIXED_USER_TAU,
                 "--agent-llm-args",json.dumps({"temperature":0.1,"max_tokens":CFG["tau_agent_tokens"]}),
                 "--user-llm-args",json.dumps({"temperature":0.1,"max_tokens":CFG["tau_user_tokens"]}),
                 "--num-trials","1","--task-split-name","base",
                 "--max-steps",str(CFG["tau_steps"]),"--max-errors","3",
                 "--max-concurrency","1","--max-retries","1","--retry-delay","2",
                 "--seed",str(SEED),"--save-to",run_name,"--auto-resume","--verbose-logs","--llm-log-mode","all"]
            if CFG["tau_tasks"] is not None:cmd+=["--num-tasks",str(CFG["tau_tasks"])]

            p=run(cmd,cwd=TAU,env=env,timeout=None);save_log(f"tau_{s['family']}_{domain}.log",p,cmd)
            if live.exists():copytree(live,archive)
            rp=archive/"results.json"
            d=parse_tau(rp) if rp.exists() else pd.DataFrame()
            n=int(d.reward.notna().sum()) if len(d) else 0
            errors=int(d.error.notna().sum()) if len(d) else 0
            if len(d):
                d["provider"]=s["provider"];d["family"]=s["family"];d["model"]=s["model"];d["domain"]=domain;tau_parts.append(d)
            st="SUPPORTED" if n>0 and errors==0 else ("PARTIAL" if n>0 else "INFRA_FAILURE")
            ck_set(key,st,n=n,errors=errors,returncode=p.returncode)
            tau_status.append({"family":s["family"],"domain":domain,"status":st,"n":n,"errors":errors})

tau=pd.concat(tau_parts,ignore_index=True) if tau_parts else pd.DataFrame()
tau_stat=pd.DataFrame(tau_status)
tau.to_csv(RESULTS/"14_tau_cases.csv",index=False);tau_stat.to_csv(RESULTS/"14_tau_status.csv",index=False)
display(tau_stat)

In [ ]:
# CELL 13 — UNIFIED EVIDENCE
rows=[]
if len(bfcl):
    for _,r in bfcl.iterrows():
        rows.append({"benchmark":"BFCL-v4","provider":r["provider"],"family":r["family"],"model":r["model"],
                     "slice":r["slice"],"metric":r["metric"],"score":float(r["score"])})
if len(dojo):
    for _,r in dojo.iterrows():
        if pd.notna(r.utility):
            rows.append({"benchmark":"AgentDojo","provider":r["provider"],"family":r["family"],"model":r["model"],
                         "slice":r["suite"],"metric":"utility","score":float(r.utility)})
        if pd.notna(r.security):
            rows.append({"benchmark":"AgentDojo","provider":r["provider"],"family":r["family"],"model":r["model"],
                         "slice":r["suite"],"metric":"security","score":float(r.security)})
if len(tau):
    for _,r in tau.iterrows():
        if pd.notna(r.reward):
            rows.append({"benchmark":"tau3","provider":r["provider"],"family":r["family"],"model":r["model"],
                         "slice":r["domain"],"metric":"reward","score":float(r.reward)})

evidence=pd.DataFrame(rows)
evidence.to_csv(RESULTS/"20_unified_evidence.csv",index=False)

if len(evidence):
    summary=(evidence.groupby(["benchmark","provider","family","model","slice","metric"],dropna=False)
             .agg(n=("score","count"),mean=("score","mean"),min=("score","min"),max=("score","max")).reset_index())
else:
    summary=pd.DataFrame()
summary.to_csv(RESULTS/"20_summary.csv",index=False)
display(summary)

In [ ]:
# CELL 14 — STRICT CLOSURE GATE
families={s["family"] for s in selected}

bf_ok=(SELFTEST or (len(bfcl_stat)>0 and families.issubset(
    set(bfcl_stat.loc[bfcl_stat.status.astype(str).str.startswith("SUPPORTED"),"family"].astype(str))
)))

dojo_ok=SELFTEST
if not SELFTEST:
    dojo_ok=True
    good=dojo_stat[dojo_stat.status.astype(str).str.startswith("SUPPORTED")]
    for fam in families:
        have=set(good.loc[good.family==fam,"suite"].astype(str))
        if not set(CFG["dojo_suites"]).issubset(have):dojo_ok=False;break

tau_ok=SELFTEST
if not SELFTEST:
    tau_ok=True
    good=tau_stat[tau_stat.status.astype(str).str.startswith("SUPPORTED")]
    for fam in families:
        have=set(good.loc[good.family==fam,"domain"].astype(str))
        if not set(CFG["tau_domains"]).issubset(have):tau_ok=False;break

claims=pd.DataFrame([
    {"claim":"BFCL-v4 external validation","status":"SELFTEST_ONLY" if SELFTEST else ("SUPPORTED" if bf_ok else "MISSING")},
    {"claim":"AgentDojo external validation","status":"SELFTEST_ONLY" if SELFTEST else ("SUPPORTED" if dojo_ok else "MISSING")},
    {"claim":"tau3 external validation","status":"SELFTEST_ONLY" if SELFTEST else ("SUPPORTED" if tau_ok else "PARTIAL" if len(tau) else "MISSING")},
    {"claim":">=3 independent model families","status":"SELFTEST_ONLY" if SELFTEST else ("SUPPORTED" if len(families)>=3 else "MISSING")},
    {"claim":"Paper external-closure gate","status":"SELFTEST_ONLY" if SELFTEST else ("SUPPORTED" if bf_ok and dojo_ok and tau_ok and len(families)>=3 else "INCOMPLETE")},
])
claims.to_csv(RESULTS/"22_claim_gate.csv",index=False)
display(claims)

In [ ]:
# CELL 15 — PAPER-INTEGRATION TABLES + SECTION
summary.to_csv(PAPER/"external_summary.csv",index=False)
summary.to_latex(PAPER/"external_summary.tex",index=False,float_format="%.4f")
claims.to_latex(PAPER/"external_claim_gate.tex",index=False)

gate=claims.loc[claims.claim=="Paper external-closure gate","status"].iloc[0]
fams=", ".join(sorted(families))

if SELFTEST:
    section="\\paragraph{External validation.} NON-PAPER SELFTEST output."
elif gate=="SUPPORTED":
    section=(
        "\\paragraph{External validation.}\n"
        "We evaluated the authorization runtime on BFCL-v4, AgentDojo, and $\\tau^3$ "
        f"using three independent model families ({fams}) across OpenAI-compatible provider interfaces. "
        "We report benchmark-native metrics separately and do not pool unlike scores. "
        "All rows in the external table originate from native benchmark outputs with evaluated cases."
    )
else:
    section=(
        "\\paragraph{External validation.}\n"
        "The external-validation matrix remains incomplete. Available benchmark-native rows are reported "
        "only as bounded transfer evidence; unsupported cross-benchmark generalization claims are omitted."
    )

(PAPER/"external_validation_section.tex").write_text(section+"\n")
print(section)

In [ ]:
# CELL 16 — BOOTSTRAP CIs + FIGURES
def boot(x,B):
    x=np.asarray(pd.Series(x).dropna(),float)
    if len(x)==0:return np.nan,np.nan,np.nan
    if len(x)==1:return float(x[0]),np.nan,np.nan
    rng=np.random.default_rng(SEED)
    m=np.array([rng.choice(x,size=len(x),replace=True).mean() for _ in range(B)])
    return float(x.mean()),float(np.quantile(m,.025)),float(np.quantile(m,.975))

cirows=[]
if len(evidence):
    for (b,f,s,m),g in evidence.groupby(["benchmark","family","slice","metric"]):
        mean,lo,hi=boot(g.score,CFG["bootstrap"])
        cirows.append({"benchmark":b,"family":f,"slice":s,"metric":m,"n":len(g),"mean":mean,"ci95_low":lo,"ci95_high":hi})
ci=pd.DataFrame(cirows)
ci.to_csv(PAPER/"external_bootstrap_ci.csv",index=False)
ci.to_latex(PAPER/"external_bootstrap_ci.tex",index=False,float_format="%.4f")

if len(summary):
    for (b,m),g in summary.groupby(["benchmark","metric"]):
        agg=g.groupby("family",as_index=False)["mean"].mean().sort_values("mean")
        fig,ax=plt.subplots(figsize=(8,max(3,0.5*len(agg)+1)))
        ax.barh(agg.family,agg["mean"]);ax.set_title(f"{b}: {m}");ax.set_xlabel(m);fig.tight_layout()
        safe=re.sub(r"[^A-Za-z0-9]+","_",f"{b}_{m}")
        fig.savefig(PAPER/f"{safe}.png",dpi=220,bbox_inches="tight")
        plt.show()

In [ ]:
# CELL 17 — PROVENANCE + FOUR FINAL ZIP FILES
manifest={
    "experiment":"NTX-MULTI-PROVIDER-PAPER-CLOSURE",
    "created_at":now(),"mode":MODE,"selftest":SELFTEST,
    "selected_models":selected,"claims":claims.to_dict("records"),"config":CFG,
}
save_json(RESULTS/"FINAL_MANIFEST.json",manifest)

# Hash non-secret artifacts.
hashes=[]
for group,folder in [("results",RESULTS),("paper",PAPER),("logs",LOGS),("environment",ENV)]:
    for p in folder.rglob("*"):
        if p.is_file():
            hashes.append({"file":f"{group}/{p.relative_to(folder)}","sha256":sha256_file(p)})
pd.DataFrame(hashes).to_csv(RESULTS/"SHA256_MANIFEST.csv",index=False)

raw_stage=BASE/"_raw";shutil.rmtree(raw_stage,ignore_errors=True);raw_stage.mkdir()
copytree(RAW,raw_stage/"raw");copytree(LOGS,raw_stage/"logs");copytree(ENV,raw_stage/"environment")

RAW_ZIP=ARCH/"NTX_MULTI_PROVIDER_RAW_DATA.zip"
RESULTS_ZIP=ARCH/"NTX_MULTI_PROVIDER_RESULTS.zip"
PAPER_ZIP=ARCH/"NTX_MULTI_PROVIDER_PAPER_INTEGRATION.zip"
MASTER_ZIP=ARCH/"NTX_MULTI_PROVIDER_PAPER_CLOSURE_MASTER.zip"

make_zip(raw_stage,RAW_ZIP);make_zip(RESULTS,RESULTS_ZIP);make_zip(PAPER,PAPER_ZIP)

master=BASE/"_master";shutil.rmtree(master,ignore_errors=True);master.mkdir()
for p in [RAW_ZIP,RESULTS_ZIP,PAPER_ZIP]:shutil.copy2(p,master/p.name)
copytree(RESULTS,master/"results");copytree(PAPER,master/"paper_integration");copytree(LOGS,master/"logs");copytree(ENV,master/"environment")
make_zip(master,MASTER_ZIP)

archive_df=pd.DataFrame([
    {"file":p.name,"size_mib":round(p.stat().st_size/1024**2,3),"sha256":sha256_file(p)}
    for p in [RAW_ZIP,RESULTS_ZIP,PAPER_ZIP,MASTER_ZIP]
])
archive_df.to_csv(ARCH/"ARCHIVE_MANIFEST.csv",index=False)
display(archive_df)

In [ ]:
# CELL 18 — FINAL DOWNLOAD
gate=claims.loc[claims.claim=="Paper external-closure gate","status"].iloc[0]
print("Closure gate:",gate)
print("Master:",MASTER_ZIP)
print("SHA256:",sha256_file(MASTER_ZIP))

try:
    from google.colab import files
    for p in [PAPER_ZIP,RESULTS_ZIP,RAW_ZIP,MASTER_ZIP]:files.download(str(p))
except Exception as e:
    print("Auto-download unavailable:",repr(e))